In [1]:
import sys
sys.path.append('..')
import pandas as pd
from src import helpers

/home/abdelnour-main/miniconda3/envs/text2kg-main-2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = "/home/abdelnour-main/Documents/Projects/Text2KG-rewrite/Text2KG/data/expertise_maps/nantes-uni-2020--Conv_Attn_null_False/dump.json"

In [3]:
raw = helpers.load_json_file(path)

In [6]:
entity_community = pd.DataFrame(raw['edges']['Entity_BelongsTo_Expertise'])
community = pd.DataFrame(raw['nodes']['Expertise'])
entities = pd.DataFrame(raw['nodes']['Entity'])

In [7]:
communities = entity_community \
    .merge(community, left_on="dst", right_on="id") \
    .merge(entities, left_on="src", right_on="id") \
    .get(['src','name_x','specificity_weighted_pagerank']) \
    .groupby('name_x').agg(list)

communities['size'] = communities['src'].apply(len)

In [8]:
import asyncio
from openai import AsyncClient
from tqdm.auto import tqdm

client = AsyncClient(
    api_key="sk-46e0b8e6cb7d4c989b13af3980c397c3",
    base_url="https://api.deepseek.com"
)

async def evaluate_community(client : AsyncClient, entities : list[str], k : int = 10) -> list[str]:

    messages = [
        {
            "role": "system",
            "content": (
                "You are a ranking system that you will be given an expertise represented by a pipe ( | ) separated list of concepts "
                f"and you are tasked with selecting {k} concepts (from the provided list) that encodes the domains present in the expertise the best. "
                f"The result should be a pipe separated list of the {k} entities ranked from the most relevant concept as the "
                "to least relevant concept (last), no additional explanation is required."
            ),
        },
        {
            "role": "user",
            "content": "|".join(entities)
        }
    ]

    response = await client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        stream=False,
        temperature=0,  # No randomness
        top_p=1,        # Disable sampling
        seed=42,        # Seed
    )

    content = response.choices[0].message.content
    entities = [entity.strip(' ') for entity in content.split('|')]

    return entities

async def evaluate(
    client : AsyncClient, 
    communities : pd.DataFrame, 
    k : int = 5,
    min_community_size : int = 15,
    batch_size : int = 5
) -> tuple[dict, list[dict]]:
    
    batch = []
    all_results = []

    metrics = {
        'top-1' : 0.0,
        'top-3' : 0.0,
        'top-5' : 0.0,
    }

    for community_label, community_entities in tqdm(communities.iterrows(), total=len(communities)):

        if len(community_entities['src']) < min_community_size:
            continue

        batch.append((
            community_label,
            list(
                map(
                    lambda x : x[0], 
                    sorted(
                        zip(
                            community_entities['src'], 
                            community_entities['specificity_weighted_pagerank']
                        ),
                        reverse=True,
                        key=lambda x : x[1]
                    )
                )
            )[:min(len(community_entities['src']), 50)]
        ))

        if len(batch) == batch_size:

            results = await asyncio.gather(*[
                evaluate_community(
                    client, 
                    entities, 
                    k
                )
                for label, entities in batch
            ])

            all_results.extend([
                {
                    "name" : name.replace(' ','_'),
                    f'top-{k}' : top_k,
                    "entities" : entities,
                }
                for (name, entities), top_k in zip(batch, results)
            ])

            batch = []

    if len(batch) > 0:

        results = await asyncio.gather(*[
            evaluate_community(
                client, 
                entities, 
                k
            )
            for label, entities in batch
        ])

        all_results.extend([
            {
                "name" : name.replace(' ','_'),
                f'top-{k}' : top_k,
                "entities" : entities
            }
            for (name, entities), top_k in zip(batch, results)
        ])

        batch = []

    for result in all_results:
        metrics['top-1'] += int(result['name'] == result[f'top-{k}'][0]) / len(all_results)
        metrics['top-3'] += int(result['name'] in result[f'top-{k}'][:3]) / len(all_results)
        metrics['top-5'] += int(result['name'] in result[f'top-{k}'][:5]) / len(all_results)

    return metrics, all_results

In [9]:
metrics, results = await evaluate(
    client,
    communities,
    min_community_size=15,
    k=10
)

100%|██████████| 796/796 [10:55<00:00,  1.21it/s]


In [10]:
metrics

{'top-1': 0.36927223719676705,
 'top-3': 0.49730458221024637,
 'top-5': 0.5646900269541801}

### With filtering / cleaning general and irrelevant entities : 

```json
    {
        'top-1': 0.3806896551724125,
        'top-3': 0.5241379310344816,
        'top-5': 0.5793103448275869
    }
```

### Without :

```json
    {
        'top-1': 0.36927223719676705,
        'top-3': 0.49730458221024637,
        'top-5': 0.5646900269541801
    }
```